# Member 3: Intermediate Modeling (ARIMA, Gradient Boosting, DM test, Feature Ablation)

**Goal:** Test whether ARIMA and Gradient Boosting can predict MSFT's next-day return better than the naive forecast (always predict 0 return).

**What is in this notebook, in order**
1. Load data and build 47 predictors (29 walk-forward splits, 1781 test days)
2. Train Gradient Boosting, ARIMA and Ridge (Ridge is re-trained here because member2_ridge_predictions.csv was empty)
3. Results table: RMSE, MAE, directional accuracy, out-of-sample R2 (Naive is the benchmark)
4. Diebold-Mariano (DM) test: is the difference from naive statistically significant?
5. Feature ablation: retrain Gradient Boosting without one predictor group at a time
6. Export predictions to outputs/member3_predictions.csv for Member 4

**How to read the tables**
- Lower RMSE and MAE are better.
- Out-of-sample R2 above 0 means better than naive (naive scores exactly 0).
- DM statistic: positive means the model is better than naive, negative means worse.
- DM p-value below 0.05 means the difference is statistically significant.

**Headline result:** No model beat the naive forecast. Ridge and Gradient Boosting were significantly worse; ARIMA was about the same.

In [1]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.ensemble import GradientBoostingRegressor
from statsmodels.tsa.arima.model import ARIMA

import tslab.config as cfg
from tslab.data import load_market_data
from tslab.features import make_dataset, check_no_lookahead
from tslab.validation import walk_forward_splits, walk_forward_predict

frame = load_market_data()
check_no_lookahead(frame)

dataset = make_dataset(frame, horizon=1).dropna(subset=["target_return"])
feature_cols = [c for c in dataset.columns if c not in ("close", "target_return", "target_close")]
X = dataset[feature_cols]
y = dataset["target_return"]

splits = list(walk_forward_splits(len(dataset), cfg.INITIAL_TRAIN, cfg.TEST_SIZE, gap=0))
print(len(dataset), "rows,", len(feature_cols), "features,", len(splits), "splits")
print(feature_cols)

Loaded 2981 trading days of MSFT, 2015-01-02 to 2026-11-09, with 7 predictor series.
2781 rows, 47 features, 29 splits
['ret_lag1', 'ret_lag2', 'ret_lag3', 'ret_lag4', 'ret_lag5', 'ret_5d', 'ret_10d', 'ret_21d', 'ret_63d', 'vol_5d', 'vol_21d', 'vol_63d', 'vol_ratio', 'range_hl', 'atr_14', 'rsi_14', 'macd_hist', 'bb_percent_b', 'bb_width', 'dist_sma_20', 'dist_sma_50', 'dist_sma_200', 'gap_overnight', 'move_intraday', 'volume_rel_20d', 'SP500_ret_1d', 'SP500_ret_5d', 'TECH_ret_1d', 'TECH_ret_5d', 'GOLD_ret_1d', 'GOLD_ret_5d', 'BRENT_ret_1d', 'BRENT_ret_5d', 'USD_ret_1d', 'USD_ret_5d', 'VIX_level', 'VIX_change_1d', 'US10Y_level', 'US10Y_change_5d', 'relative_to_SP500_5d', 'is_mon', 'is_tue', 'is_wed', 'is_thu', 'is_fri', 'month_sin', 'month_cos']


In [2]:
for i in range(0, len(feature_cols), 6):
    print(feature_cols[i:i+6])

['ret_lag1', 'ret_lag2', 'ret_lag3', 'ret_lag4', 'ret_lag5', 'ret_5d']
['ret_10d', 'ret_21d', 'ret_63d', 'vol_5d', 'vol_21d', 'vol_63d']
['vol_ratio', 'range_hl', 'atr_14', 'rsi_14', 'macd_hist', 'bb_percent_b']
['bb_width', 'dist_sma_20', 'dist_sma_50', 'dist_sma_200', 'gap_overnight', 'move_intraday']
['volume_rel_20d', 'SP500_ret_1d', 'SP500_ret_5d', 'TECH_ret_1d', 'TECH_ret_5d', 'GOLD_ret_1d']
['GOLD_ret_5d', 'BRENT_ret_1d', 'BRENT_ret_5d', 'USD_ret_1d', 'USD_ret_5d', 'VIX_level']
['VIX_change_1d', 'US10Y_level', 'US10Y_change_5d', 'relative_to_SP500_5d', 'is_mon', 'is_tue']
['is_wed', 'is_thu', 'is_fri', 'month_sin', 'month_cos']


In [3]:
gb_pred = walk_forward_predict(
    lambda: GradientBoostingRegressor(n_estimators=200, max_depth=2,
                                      learning_rate=0.05, subsample=0.8, random_state=42),
    X, y, splits, standardize=False)
print(gb_pred.notna().sum(), "out-of-sample predictions")

1781 out-of-sample predictions


In [4]:
arima_pred = pd.Series(np.nan, index=dataset.index)

for train_pos, test_pos in splits:
    res = ARIMA(y.iloc[train_pos].values, order=(1, 0, 1), trend="c").fit()
    res_all = res.apply(y.iloc[: test_pos[-1] + 1].values)
    pred = res_all.predict(start=test_pos[0], end=test_pos[-1])
    arima_pred.iloc[test_pos] = pred

print(arima_pred.notna().sum(), "out-of-sample predictions")

c:\ts\.venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


1781 out-of-sample predictions


In [5]:
from sklearn.linear_model import Ridge
ridge = walk_forward_predict(lambda: Ridge(alpha=10.0), X, y, splits, standardize=True)
print("Ridge:", ridge.notna().sum())

Ridge: 1781


Ridge is re-trained in this notebook (alpha = 10, standardized features, same walk-forward splits as the other models) because the saved file member2_ridge_predictions.csv was empty.

In [6]:
preds = pd.DataFrame({"Naive (0 return)": 0.0, "Ridge": ridge,
                      "ARIMA": arima_pred, "GradientBoosting": gb_pred})
mask = preds.drop(columns="Naive (0 return)").notna().all(axis=1)
actual = y[mask]
preds = preds[mask]

def metrics(a, p):
    e = a - p
    return {"RMSE": np.sqrt((e**2).mean()), "MAE": e.abs().mean(),
            "Directional acc.": (np.sign(a) == np.sign(p)).mean() if p.abs().sum() > 0 else np.nan,
            "OOS R2": 1 - (e**2).sum() / (a**2).sum()}

table = pd.DataFrame({m: metrics(actual, preds[m]) for m in preds.columns}).T
print(len(actual), "test days")
table

1781 test days


,RMSE,MAE,Directional acc.,OOS R2
Naive (0 return),0.018854,0.013024,NaN,0.000000
Ridge,0.019376,0.013605,0.507019,-0.056230
ARIMA,0.018853,0.013172,0.522740,0.000105
GradientBoosting,0.019981,0.013900,0.517125,-0.123221


In [7]:
def dm_test(actual, naive, model, power=2):
    """Positive stat => model has LOWER loss than naive. h=1, Harvey-Leybourne-Newbold correction."""
    d = np.abs(actual - naive) ** power - np.abs(actual - model) ** power
    n = len(d)
    stat = d.mean() / np.sqrt(d.var(ddof=0) / n) * np.sqrt((n - 1) / n)
    p = 2 * (1 - stats.t.cdf(abs(stat), df=n - 1))
    return stat, p

rows = {}
for m in ["Ridge", "ARIMA", "GradientBoosting"]:
    s2, p2 = dm_test(actual, preds["Naive (0 return)"], preds[m], power=2)
    s1, p1 = dm_test(actual, preds["Naive (0 return)"], preds[m], power=1)
    rows[m] = {"DM stat (sq. err)": s2, "p-value (sq.)": p2, "DM stat (abs err)": s1, "p-value (abs)": p1}
dm_table = pd.DataFrame(rows).T
dm_table

,DM stat (sq. err),p-value (sq.),DM stat (abs err),p-value (abs)
Ridge,-2.561976,0.010490,-4.406598,1.112664e-05
ARIMA,0.007453,0.994055,-1.766632,7.746131e-02
GradientBoosting,-3.794524,0.000153,-5.273740,1.499868e-07


In [8]:
s, p = dm_test(actual, preds["Naive (0 return)"], actual, power=2)
print(s, p)

12.976877663873495 0.0


### Diebold-Mariano test results

We tested each model against the naive forecast (predicting a 0 return). A positive DM statistic means the model is better than naive; a negative one means it is worse. We confirmed this by testing a perfect forecast, which gave a strongly positive statistic (12.98).

- **Ridge:** significantly worse than naive (p = 0.010 for squared error, p < 0.001 for absolute error).
- **Gradient Boosting:** significantly worse than naive (p = 0.0002 for squared error, p < 0.001 for absolute error).
- **ARIMA:** not significantly different from naive (p = 0.99 for squared error, p = 0.077 for absolute error). Its forecasts are close to 0, so it behaves like the naive forecast.

**Conclusion:** None of our models significantly beat the naive forecast on this test period. We report this honestly, as daily stock returns are very hard to predict.

Note: Directional accuracy is not defined for the naive forecast because it always predicts 0. ARIMA's RMSE is virtually identical to naive and its out-of-sample R2 is about 0, so it adds no real value over naive.

In [9]:
export = preds.copy()
export.insert(0, "actual", actual)
export.to_csv(cfg.OUTPUT_DIR / "member3_predictions.csv")
print(export.shape)

(1781, 5)


In [10]:
cols = list(X.columns)
groups = {}
groups["Past returns"] = [c for c in cols if c.startswith("ret_")]
groups["Volatility"] = [c for c in cols if c in ("vol_5d","vol_21d","vol_63d","vol_ratio","range_hl","atr_14")]
groups["Technical indicators"] = [c for c in cols if c in ("rsi_14","macd_hist","bb_percent_b","bb_width") or c.startswith("dist_sma")]
groups["Market and other assets"] = [c for c in cols if c.startswith(("SP500_","TECH_","GOLD_","BRENT_","USD_","relative_to_SP500"))]
groups["VIX and interest rates"] = [c for c in cols if c.startswith(("VIX_","US10Y_"))]
groups["Calendar"] = [c for c in cols if c.startswith(("is_","month_"))]
used = sum(groups.values(), [])
groups["Price gaps and volume"] = [c for c in cols if c not in used]
for g, cs in groups.items():
    print(g, len(cs), cs)
print(sum(len(v) for v in groups.values()), "of", len(cols))

Past returns 9 ['ret_lag1', 'ret_lag2', 'ret_lag3', 'ret_lag4', 'ret_lag5', 'ret_5d', 'ret_10d', 'ret_21d', 'ret_63d']
Volatility 6 ['vol_5d', 'vol_21d', 'vol_63d', 'vol_ratio', 'range_hl', 'atr_14']
Technical indicators 7 ['rsi_14', 'macd_hist', 'bb_percent_b', 'bb_width', 'dist_sma_20', 'dist_sma_50', 'dist_sma_200']
Market and other assets 11 ['SP500_ret_1d', 'SP500_ret_5d', 'TECH_ret_1d', 'TECH_ret_5d', 'GOLD_ret_1d', 'GOLD_ret_5d', 'BRENT_ret_1d', 'BRENT_ret_5d', 'USD_ret_1d', 'USD_ret_5d', 'relative_to_SP500_5d']
VIX and interest rates 4 ['VIX_level', 'VIX_change_1d', 'US10Y_level', 'US10Y_change_5d']
Calendar 7 ['is_mon', 'is_tue', 'is_wed', 'is_thu', 'is_fri', 'month_sin', 'month_cos']
Price gaps and volume 3 ['gap_overnight', 'move_intraday', 'volume_rel_20d']
47 of 47


In [11]:
def run_gb(use_cols):
    return walk_forward_predict(
        lambda: GradientBoostingRegressor(n_estimators=200, max_depth=2,
                                          learning_rate=0.05, subsample=0.8, random_state=42),
        X[use_cols], y, splits, standardize=False)

abl = {"Naive (0 return)": pd.Series(0.0, index=dataset.index),
       "Full model (all features)": gb_pred}
for g, cs in groups.items():
    abl[f"Without {g}"] = run_gb([c for c in cols if c not in cs])
    print("done:", g)

done: Past returns
done: Volatility
done: Technical indicators
done: Market and other assets
done: VIX and interest rates
done: Calendar
done: Price gaps and volume


In [12]:
abl_df = pd.DataFrame(abl)
ok = abl_df.drop(columns="Naive (0 return)").notna().all(axis=1)
a = y[ok]
ab = abl_df[ok]
abl_table = pd.DataFrame({k: metrics(a, ab[k]) for k in ab.columns}).T
abl_table["RMSE change vs full"] = abl_table["RMSE"] - abl_table.loc["Full model (all features)", "RMSE"]
res = {k: dm_test(a, ab["Naive (0 return)"], ab[k], power=2) for k in ab.columns if k != "Naive (0 return)"}
abl_table["DM stat vs naive"] = pd.Series({k: v[0] for k, v in res.items()})
abl_table["DM p-value"] = pd.Series({k: v[1] for k, v in res.items()})
abl_table

,RMSE,MAE,Directional acc.,OOS R2,RMSE change vs full,DM stat vs naive,DM p-value
Naive (0 return),0.018854,0.013024,NaN,0.000000,-0.001128,NaN,NaN
Full model (all features),0.019981,0.013900,0.517125,-0.123221,0.000000,-3.794524,0.000153
Without Past returns,0.019709,0.013796,0.513195,-0.092759,-0.000273,-3.279441,0.001060
Without Volatility,0.019846,0.013861,0.509826,-0.108094,-0.000135,-3.362487,0.000789
Without Technical indicators,0.019724,0.013717,0.508703,-0.094464,-0.000257,-3.394903,0.000702
Without Market and other assets,0.019921,0.013898,0.514318,-0.116384,-0.000061,-3.480264,0.000513
Without VIX and interest rates,0.019843,0.013753,0.514879,-0.107702,-0.000139,-3.769638,0.000169
Without Calendar,0.019935,0.013872,0.510387,-0.117954,-0.000047,-3.576265,0.000358
Without Price gaps and volume,0.019884,0.013803,0.522179,-0.112251,-0.000098,-3.410210,0.000664


Feature ablation: We retrained Gradient Boosting after removing one predictor group at a time. Removing any single group lowered the RMSE slightly (by 0.00005 to 0.00027), so no group clearly helped the model; the largest improvements came from dropping past returns (-0.00027) and technical indicators (-0.00026). However, these differences are small (under 1.5% of the RMSE), and even the best version (without past returns, RMSE 0.01971) is still worse than the naive forecast (0.01885), with every version significantly worse in the DM test (all p < 0.002). So we cannot identify a predictor group that gives real forecasting skill, which is consistent with daily stock returns being very hard to predict.